In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.gold_KPIs;

1. Monthly Revenue Trend
Merge Sales, Products, Exchange Rates. Calculate total revenue USD by month (2024).


In [0]:
%sql
SELECT 
  YEAR(order_date) AS year,
  MONTH(order_date) AS month,
  ROUND(SUM(revenue_usd), 2) AS revenue_usd
FROM electronics_retailer_clg.gold.fact_sales
WHERE YEAR(order_date) = 2021
GROUP BY YEAR(order_date), MONTH(order_date)
ORDER BY month;

2. Peak Month Analysis
From Q1, identify top 3 revenue months. Calculate % of annual total.


In [0]:
%sql
WITH monthly AS (
  SELECT 
    MONTH(order_date) AS month,
    SUM(revenue_usd) AS revenue_usd
  FROM electronics_retailer_clg.gold.fact_sales
  WHERE YEAR(order_date) = 2021
  GROUP BY MONTH(order_date)
),
total AS (
  SELECT SUM(revenue_usd) AS total_revenue
  FROM electronics_retailer_clg.gold.fact_sales
  WHERE YEAR(order_date) = 2021
)

SELECT 
  m.month,
  ROUND(m.revenue_usd, 2) AS revenue_usd,
  ROUND((m.revenue_usd / t.total_revenue) * 100, 2) AS pct_of_total
FROM monthly m
CROSS JOIN total t
ORDER BY m.revenue_usd DESC
LIMIT 3;

3. Holiday Drivers
For Q2 peak months, show top 3 product categories driving revenue.

In [0]:
%sql
-- Holiday Drivers (Q2 Peak Months)

WITH peak_months AS (
  SELECT 
    MONTH(order_date) AS month
  FROM electronics_retailer_clg.gold.fact_sales
  WHERE YEAR(order_date) = 2021
    AND QUARTER(order_date) = 2
  GROUP BY MONTH(order_date)
  ORDER BY SUM(revenue_usd) DESC
  LIMIT 3
),

category_sales AS (
  SELECT 
    product_category,
    SUM(revenue_usd) AS revenue
  FROM electronics_retailer_clg.gold.fact_sales
  WHERE YEAR(order_date) = 2021
    AND MONTH(order_date) IN (SELECT month FROM peak_months)
  GROUP BY product_category
),

total AS (
  SELECT SUM(revenue) AS total_revenue FROM category_sales
)

SELECT 
  c.product_category,
  ROUND(c.revenue, 2) AS peak_month_revenue,
  ROUND((c.revenue / t.total_revenue) * 100, 2) AS pct_of_peak_total
FROM category_sales c
CROSS JOIN total t
ORDER BY peak_month_revenue DESC
LIMIT 3;

4. Delivery Performance
Calculate overall avg delivery time across all orders.
Output: Average_Days | Total_Orders


In [0]:
%sql
SELECT 
  ROUND(AVG(DATEDIFF(delivery_date, order_date)), 2) AS average_days,
  COUNT(*) AS total_orders
FROM electronics_retailer_clg.gold.fact_sales
WHERE delivery_date IS NOT NULL;

5. Country Delivery Issues
Avg delivery time by store country. Show slowest 5 countries.


In [0]:
%sql
SELECT 
  store_country,
  ROUND(AVG(DATEDIFF(delivery_date, order_date)), 2) AS avg_days,
  COUNT(*) AS order_count,
  ROUND(percentile_approx(DATEDIFF(delivery_date, order_date),0.5),2) AS median_days
FROM electronics_retailer_clg.gold.fact_sales
WHERE delivery_date IS NOT NULL
GROUP BY store_country
ORDER BY avg_days DESC
LIMIT 5;

6. Channel Performance
AOV (revenue/orders) by Online vs In-Store across continents (Online = StoreKey.isna()).

In [0]:
%sql
-- Channel Performance: AOV by Online vs Store

WITH base AS (
  SELECT 
    continent,
    order_number,
    revenue_usd,
    CASE 
      WHEN storekey IS NULL OR storekey = 0 THEN 'online'
      ELSE 'store'
    END AS channel
  FROM electronics_retailer_clg.gold.fact_sales
  WHERE continent IS NOT NULL
),

agg AS (
  SELECT
    continent,
    channel,
    COUNT(DISTINCT order_number) AS orders,
    SUM(revenue_usd) AS revenue
  FROM base
  GROUP BY continent, channel
)

SELECT
  continent,

  -- AOV
  ROUND(
    SUM(CASE WHEN channel='online' THEN revenue END) /
    SUM(CASE WHEN channel='online' THEN orders END), 2
  ) AS aov_online,

  ROUND(
    SUM(CASE WHEN channel='store' THEN revenue END) /
    SUM(CASE WHEN channel='store' THEN orders END), 2
  ) AS aov_store,

  -- Order counts
  SUM(CASE WHEN channel='online' THEN orders END) AS online_orders,
  SUM(CASE WHEN channel='store' THEN orders END) AS store_orders

FROM agg
GROUP BY continent
ORDER BY continent;

7. Volume Leaders
Top 5 product categories by total units sold.

In [0]:
%sql
WITH total AS (
  SELECT SUM(quantity) AS total_units 
  FROM electronics_retailer_clg.gold.fact_sales
)

SELECT 
  RANK() OVER (ORDER BY SUM(quantity) DESC) AS rank,
  product_category,
  SUM(quantity) AS units_sold,
  ROUND((SUM(quantity)/t.total_units)*100,2) AS pct_of_total_units
FROM electronics_retailer_clg.gold.fact_sales
CROSS JOIN total t
GROUP BY product_category, t.total_units
ORDER BY units_sold DESC
LIMIT 5;


8. Revenue Leaders
Top 5 product categories by total revenue USD.

In [0]:
%sql
WITH total AS (
  SELECT SUM(revenue_usd) AS total_rev 
  FROM electronics_retailer_clg.gold.fact_sales
)

SELECT 
  RANK() OVER (ORDER BY SUM(revenue_usd) DESC) AS rank,
  product_category,
  ROUND(SUM(revenue_usd),2) AS revenue_usd,
  ROUND((SUM(revenue_usd)/t.total_rev)*100,2) AS pct_of_total_revenue
FROM electronics_retailer_clg.gold.fact_sales
CROSS JOIN total t
GROUP BY product_category, t.total_rev
ORDER BY revenue_usd DESC
LIMIT 5;

9. Customer Profile
Customer count and spending by Continent × Gender.


In [0]:
%sql
SELECT 
  continent,
  gender,
  COUNT(DISTINCT customerkey) AS customer_count,
  ROUND(SUM(revenue_usd),2) AS total_spend_usd,
  ROUND(SUM(revenue_usd)/COUNT(DISTINCT customerkey),2) AS avg_spend_per_cust
FROM electronics_retailer_clg.gold.fact_sales
WHERE customerkey IS NOT NULL
GROUP BY continent, gender;

10. Customer Loyalty
Repeat customer rate (% with 2+ orders) by continent.


In [0]:
%sql
WITH cust_orders AS (
  SELECT 
    customerkey,
    continent,
    COUNT(DISTINCT order_number) AS orders
  FROM electronics_retailer_clg.gold.fact_sales
  WHERE customerkey IS NOT NULL
  GROUP BY customerkey, continent
)

SELECT 
  continent,
  ROUND(
    (SUM(CASE WHEN orders>=2 THEN 1 ELSE 0 END) /
     COUNT(*)) * 100, 2
  ) AS repeat_rate_pct,
  COUNT(*) AS unique_customers,
  SUM(CASE WHEN orders>=2 THEN 1 ELSE 0 END) AS repeat_customers
FROM cust_orders
GROUP BY continent;